# 02b — Modélisation généralisée : les 5 attributs de formulation

Même pipeline que pour `contains_fragrance` et `contains_silicones`, appliqué automatiquement aux 5 cibles :
1. Catégorie seule (référence, sans image)
2. Features manuelles (couleur/contraste)
3. Embeddings CLIP seuls (régularisé, C=0.1)
4. Embeddings CLIP + catégorie (régularisé, C=0.1)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGETS = [
    "contains_fragrance",
    "contains_drying_alcohol",
    "contains_parabens",
    "contains_sulfates",
    "contains_silicones",
]

# Chargement 
df_meta = pd.read_parquet("../data/processed/dataset_clean.parquet")
df_manual = pd.read_parquet("../data/processed/features_manual.parquet")
embeddings = np.load("../data/processed/embeddings_clip.npy")
embedding_ids = np.load("../data/processed/embeddings_ids.npy")

df_emb = pd.DataFrame(embeddings, columns=[f"clip_{i}" for i in range(embeddings.shape[1])])
df_emb["id"] = embedding_ids

df_full = df_meta.merge(df_manual, on="id").merge(df_emb, on="id")

category_dummies = pd.get_dummies(df_full["category"], prefix="cat")
df_full = pd.concat([df_full, category_dummies], axis=1)
cat_cols = category_dummies.columns.tolist()

manual_cols = ["dominant_r", "dominant_g", "dominant_b", "second_r", "second_g", "second_b",
               "contrast", "brightness", "saturation", "aspect_ratio"]
clip_cols = [c for c in df_full.columns if c.startswith("clip_")]

print(f"Dataset prêt : {df_full.shape}")

## Fonction de pipeline réutilisable

In [ ]:
def run_pipeline_for_target(target, df_full):
    y = df_full[target].astype(int)

    idx_train, idx_test = train_test_split(
        df_full.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    y_train, y_test = y.loc[idx_train], y.loc[idx_test]

    results = {"target": target, "pct_true": y.mean() * 100}

    # 1. Catégorie seule
    X_cat_train = df_full.loc[idx_train, cat_cols]
    X_cat_test = df_full.loc[idx_test, cat_cols]
    m = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    m.fit(X_cat_train, y_train)
    results["auc_category_only"] = roc_auc_score(y_test, m.predict_proba(X_cat_test)[:, 1])

    # 2. Features manuelles
    X_man_train = df_full.loc[idx_train, manual_cols]
    X_man_test = df_full.loc[idx_test, manual_cols]
    m = GradientBoostingClassifier(random_state=RANDOM_STATE)
    m.fit(X_man_train, y_train)
    results["auc_manual"] = roc_auc_score(y_test, m.predict_proba(X_man_test)[:, 1])

    # 3. CLIP seul (régularisé)
    X_clip_train = df_full.loc[idx_train, clip_cols]
    X_clip_test = df_full.loc[idx_test, clip_cols]
    scaler = StandardScaler()
    X_clip_train_s = scaler.fit_transform(X_clip_train)
    X_clip_test_s = scaler.transform(X_clip_test)
    m = LogisticRegression(max_iter=2000, C=0.1, random_state=RANDOM_STATE)
    m.fit(X_clip_train_s, y_train)
    results["auc_clip"] = roc_auc_score(y_test, m.predict_proba(X_clip_test_s)[:, 1])

    # 4. CLIP + catégorie (régularisé)
    X_ctrl_train = df_full.loc[idx_train, clip_cols + cat_cols]
    X_ctrl_test = df_full.loc[idx_test, clip_cols + cat_cols]
    scaler2 = StandardScaler()
    X_ctrl_train_s = scaler2.fit_transform(X_ctrl_train)
    X_ctrl_test_s = scaler2.transform(X_ctrl_test)
    m = LogisticRegression(max_iter=2000, C=0.1, random_state=RANDOM_STATE)
    m.fit(X_ctrl_train_s, y_train)
    results["auc_clip_plus_category"] = roc_auc_score(y_test, m.predict_proba(X_ctrl_test_s)[:, 1])

    return results

## Exécution sur les 5 attributs

In [ ]:
all_results = []
for target in TARGETS:
    print(f"Traitement de {target}...")
    res = run_pipeline_for_target(target, df_full)
    all_results.append(res)

df_results = pd.DataFrame(all_results)
df_results = df_results.round(3)
df_results

## Visualisation comparative

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(TARGETS))
width = 0.2

ax.bar(x - 1.5*width, df_results["auc_category_only"], width, label="Catégorie seule")
ax.bar(x - 0.5*width, df_results["auc_manual"], width, label="Features manuelles")
ax.bar(x + 0.5*width, df_results["auc_clip"], width, label="CLIP seul")
ax.bar(x + 1.5*width, df_results["auc_clip_plus_category"], width, label="CLIP + catégorie")

ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="Hasard")
ax.set_xticks(x)
ax.set_xticklabels([t.replace("contains_", "") for t in TARGETS], rotation=15)
ax.set_ylabel("AUC-ROC")
ax.set_title("Comparaison des modèles sur les 5 attributs de formulation")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.set_ylim(0.3, 1.0)

plt.tight_layout()
plt.show()

## Synthèse à retenir

Pour chaque attribut, comparer :
- **CLIP seul vs Hasard (0.5)** : y a-t-il un signal visuel du tout ?
- **CLIP seul vs Catégorie seule** : le signal visuel est-il compétitif avec la simple connaissance de la catégorie ?
- **CLIP + catégorie vs Catégorie seule** : l'image ajoute-t-elle une information marginale une fois la catégorie connue ?

Ces trois comparaisons, répétées sur 5 attributs, permettent une discussion nuancée plutôt qu'un simple "ça marche / ça marche pas" — utile pour la partie discussion du rapport.